<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/CreateAgent/ipynb/1.1_prompting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Basic prompting

In [ ]:
%%bash
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core langchain-community \
    langchain-ollama

In [1]:
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv

_ = load_dotenv(dotenv_path=".env", override=True)

In [2]:
import os
from langchain.chat_models import init_chat_model

llm = init_chat_model(
model="qwen3.5:cloud",
model_provider="ollama",
base_url="https://ollama.com",
api_key=os.environ["OLLAMA_API_KEY"],
temperature=0
)

In [3]:
import time
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(model=llm)

start_time = time.time()
question = HumanMessage(content="""
What's the capital of the moon?
""")
response = agent.invoke(input={"messages": [question]})
end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 11.24s
================================ Human Message =================================


What's the capital of the moon?

================================== Ai Message ==================================

The Moon does not have a capital.

This is because the Moon is not a country or a sovereign nation. It has no government, and under the Outer Space Treaty of 1967, no nation can claim ownership or sovereignty over it. Additionally, there are no permanent human settlements on the Moon yet.


In [4]:
import time
from langchain.agents import create_agent
from langchain.messages import HumanMessage

system_prompt = """
You are a science fiction writer, create a capital city at the users request.
"""

scifi_agent = create_agent(
model=llm,
system_prompt=system_prompt
)

start_time = time.time()
question = HumanMessage(content="""
What's the capital of the moon?
""")
response = scifi_agent.invoke(input={"messages": [question]})
end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 39.36s
================================ Human Message =================================


What's the capital of the moon?

================================== Ai Message ==================================

In our current reality, the Moon belongs to no single nation and thus has no official capital. However, in the universe of my latest speculative fiction series, *The Celestial Accord*, I have designed a seat of power for the Lunar Federation.

Allow me to introduce you to **Armstrong Citadel**.

### **Location: The Shackleton Crater (Lunar South Pole)**
Choosing the South Pole was strategic. The rim of the Shackleton Crater receives near-permanent sunlight, providing endless solar energy, while the depths of the crater hold permanently shadowed regions rich in water ice. Armstrong Citadel bridges these two zones.

### **Architecture & Design**
The city is not built *on* the surface, but *within* it.
*   **The Regolith Shield:** The city is capped by a transparent aluminum-

## Few-shot examples

In [5]:
import time
from langchain.agents import create_agent
from langchain.messages import HumanMessage

system_prompt = """
You are a science fiction writer, create a space capital city
at the users request.

User: What is the capital of mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia
"""

scifi_agent = create_agent(
model=llm,
system_prompt=system_prompt
)

start_time = time.time()
question = HumanMessage(content="""
What's the capital of the moon?
""")
response = scifi_agent.invoke(input={"messages": [question]})
end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 53.56s
================================ Human Message =================================


What's the capital of the moon?

================================== Ai Message ==================================

Lunovia


## Structured prompts

In [6]:
import time
from langchain.agents import create_agent
from langchain.messages import HumanMessage

system_prompt = """
You are a science fiction writer, create a space capital city
at the users request.

Please keep to the below structure.

Name: The name of the capital city

Location: Where it is based

Vibe: 2-3 words to describe its vibe

Economy: Main industries
"""

scifi_agent = create_agent(
model=llm,
system_prompt=system_prompt
)

start_time = time.time()
question = HumanMessage(content="""
What's the capital of the moon?
""")
response = scifi_agent.invoke(input={"messages": [question]})
end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 20.58s
================================ Human Message =================================


What's the capital of the moon?

================================== Ai Message ==================================

Name: Artemis Reach

Location: Shackleton Crater, Lunar South Pole

Vibe: Sterile, Gleaming, Exclusive

Economy: Helium-3 Extraction, Federal Administration, Orbital Shipbuilding


## Structured output

In [7]:
import time
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel

class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

agent = create_agent(
    model=llm,
    system_prompt="""
    You are a science fiction writer.
    Create a capital city at the users request.
    """,
    response_format=CapitalInfo.model_json_schema()
)

question = HumanMessage(content="""
What is the capital of The Moon?
""")

start_time = time.time()
response = agent.invoke(input={"messages": [question]})
end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 4.97s seconds
================================ Human Message =================================


What is the capital of The Moon?

================================== Ai Message ==================================
Tool Calls:
  CapitalInfo (3895e9d8-055b-472d-831f-cb8e79f3b9a5)
 Call ID: 3895e9d8-055b-472d-831f-cb8e79f3b9a5
  Args:
    name: Selenea Prime
    location: Shackleton Crater at the Lunar South Pole
    vibe: Futuristic metropolis with crystalline domes, zero-gravity plazas, and shimmering solar arrays that glow perpetually in the eternal sunlight
    economy: Helium-3 mining, lunar manufacturing, space tourism, and interplanetary research headquarters
================================= Tool Message =================================
Name: CapitalInfo

Returning structured response: {'name': 'Selenea Prime', 'location': 'Shackleton Crater at the Lunar South Pole', 'vibe': 'Futuristic metropolis with crystalline domes, zero-gravity plazas, and shimmering solar arrays 

In [8]:
capital_info = response["structured_response"]
capital_name = capital_info["name"]
capital_location = capital_info["location"]

print(f"{capital_name} is a city located at {capital_location}")

Selenea Prime is a city located at Shackleton Crater at the Lunar South Pole
